Data exploration of DOT18,19 and 20 datasets.

Connected to DataWA's WFS. Identified the relavant dataset names for DOT18, 19 and 20.

In [99]:
# imports
from owslib.wfs import WebFeatureService
import requests
import geopandas as gpd
from io import BytesIO

Create a WFS object first with the URL.

In [46]:
WFS_URL = "https://public-services.slip.wa.gov.au/public/services/SLIP_Public_Services/Infrastructure_and_Utilities_WFS/MapServer/WFSServer"

wfs = WebFeatureService(url=WFS_URL, version='2.0.0')

Next, we need to explore the metadata of the WFS service offered by DataWA. We will also see the operations available for us to use.

In [47]:
print(wfs.identification.type)
print(wfs.identification.version)
print(wfs.identification.title)
print(wfs.identification.abstract)

operations = []
for i in wfs.operations:
    operations.append(i)
operations

WFS
2.0.0
WFS
None


[<owslib.ows.OperationsMetadata GetCapabilities at 0x114a7e190>,
 <owslib.ows.OperationsMetadata DescribeFeatureType at 0x114a7e120>,
 <owslib.ows.OperationsMetadata GetPropertyValue at 0x114a7e510>,
 <owslib.ows.OperationsMetadata GetFeature at 0x114a7e580>,
 <owslib.ows.OperationsMetadata GetGmlObject at 0x114a7e2e0>,
 <owslib.ows.OperationsMetadata ListStoredQueries at 0x114a7e040>,
 <owslib.ows.OperationsMetadata DescribeStoredQueries at 0x114a7e200>,
 <owslib.ows.OperationsMetadata ImplementsBasicWFS at 0x114a7cde0>,
 <owslib.ows.OperationsMetadata ImplementsTransactionalWFS at 0x114a7d9b0>,
 <owslib.ows.OperationsMetadata ImplementsLockingWFS at 0x114a7e350>,
 <owslib.ows.OperationsMetadata KVPEncoding at 0x114a7e3c0>,
 <owslib.ows.OperationsMetadata XMLEncoding at 0x114a7e430>,
 <owslib.ows.OperationsMetadata SOAPEncoding at 0x114a7e4a0>,
 <owslib.ows.OperationsMetadata ImplementsInheritance at 0x114a7e5f0>,
 <owslib.ows.OperationsMetadata ImplementsRemoteResolve at 0x114a7e660>

In [48]:
#what are the output formats available for getfeature request? This is our download options 
wfs.getOperationByName('GetFeature').parameters

{'resultType': {'values': ['results', 'hits']},
 'outputFormat': {'values': ['GML32',
   'GML32+ZIP',
   'GML32+GZIP',
   'application/gml+xml; version=3.2',
   'GML3',
   'GML3+ZIP',
   'GML3+GZIP',
   'text/xml; subtype=gml/3.1.1',
   'GML2',
   'GML2+ZIP',
   'GML2+GZIP',
   'text/xml; subtype=gml/2.1.2',
   'GEOJSON',
   'GEOJSON+ZIP',
   'GEOJSON+GZIP',
   'ESRIGEOJSON',
   'ESRIGEOJSON+ZIP',
   'ESRIGEOJSON+GZIP',
   'KML',
   'application/vnd.google-earth.kml xml',
   'application/vnd.google-earth.kml+xml',
   'KMZ',
   'application/vnd.google-earth.kmz',
   'SHAPE+ZIP',
   'CSV',
   'CSV+ZIP',
   'CSV+GZIP',
   'Geopackage',
   'Geopackage+ZIP']},
 'resolve': {'values': ['none', 'local']}}

In [49]:
for dataset in wfs.contents:
    print(dataset)
# Will help us investigate and identify dataset layer names of DOT18,19 and 20.
# esri:Coastal_Infrastructure_Point__DOT-018_
# esri:Coastal_Infrastructure_Line__DOT-019_
# esri:Coastal_Infrastructure_DOT__DOT-020_

esri:Water_Meter__WCORP-006_
esri:Water_Valve__WCORP-299_
esri:Water_Pipe__WCORP-002_
esri:Harvey_Water_Irrigation_Districts__HARWA-002_
esri:Harvey_Water_Pipelines__HARWA-001_
esri:Harvey_Water_Additional_Works__HARWA-004_
esri:DFES_Stations__DFES-023_
esri:Sewer_Manhole__WCORP-026_
esri:Drain_Manhole__WCORP-291_
esri:Water_Hydrant__WCORP-070_
esri:Water_Standpipe__WCORP-071_
esri:Water_Treatment_Plant__WCORP-072_
esri:Water_Bore__WCORP-073_
esri:Water_Tank__WCORP-076_
esri:Water_Pump_Station__WCORP-077_
esri:Drainage_Observation_Bore__WCORP-081_
esri:Drainage_Pump_Station__WCORP-082_
esri:Sewer_Pump_Station__WCORP-295_
esri:Sewer_Treatment_Plant__WCORP-296_
esri:Coastal_Infrastructure_Point__DOT-018_
esri:Sewer_Gravity_Pipe__WCORP-068_
esri:Sewer_Pressure_Main__WCORP-069_
esri:Drainage_Rising_Main__WCORP-074_
esri:Drainage_Gravity_Pipe__WCORP-080_
esri:Drainage_Open_Channel__WCORP-083_
esri:Drain_Inlet__WCORP-290_
esri:Irrigation_Gravity_Pipe__WCORP-292_
esri:Irrigation_Open_Channel_

Next, let us investigate the individual datasets to identify the type of structures present in each dataset. To do so, I have to first download the datasets and save as geojson files.

In [64]:
layer_names = ["esri:Coastal_Infrastructure_Point__DOT-018_", "esri:Coastal_Infrastructure_Line__DOT-019_", "esri:Coastal_Infrastructure_DOT__DOT-020_"]
print(wfs.contents[layer_names[0]].title)
wfs.get_schema(layer_names[0])

Coastal_Infrastructure_Point__DOT-018_


{'properties': {'objectid': 'int',
  'agencyid': 'double',
  'agencyname': 'string',
  'structype': 'string',
  'assetgroup': 'string',
  'assetcode': 'string',
  'assetdesc': 'string',
  'lga': 'string',
  'portauth': 'string',
  'sourcedate': 'dateTime',
  'source': 'string',
  'conmat': 'string',
  'constrdate': 'dateTime',
  'status': 'string',
  'editdate': 'dateTime',
  'techdraw': 'string',
  'refcomm': 'string',
  'reflink': 'string',
  'ci_id': 'string',
  'SourceDate_txt': 'string',
  'EditDate_txt': 'string'},
 'required': ['objectid'],
 'geometry': 'Point',
 'geometry_column': 'shape'}

In [65]:
print(wfs.contents[layer_names[1]].title)
wfs.get_schema(layer_names[1])


Coastal_Infrastructure_Line__DOT-019_


{'properties': {'objectid': 'int',
  'agencyid': 'double',
  'agencyname': 'string',
  'structype': 'string',
  'assetgroup': 'string',
  'assetcode': 'string',
  'assetdesc': 'string',
  'lga': 'string',
  'portauth': 'string',
  'sourcedate': 'dateTime',
  'source': 'string',
  'conmat': 'string',
  'constrdate': 'dateTime',
  'status': 'string',
  'editdate': 'dateTime',
  'techdraw': 'string',
  'refcomm': 'string',
  'reflink': 'string',
  'ci_id': 'string',
  'SourceDate_txt': 'string',
  'EditDate_txt': 'string'},
 'required': ['objectid'],
 'geometry': None}

In [67]:
print(wfs.contents[layer_names[0]].title)
wfs.get_schema(layer_names[2])

Coastal_Infrastructure_Point__DOT-018_


{'properties': {'objectid': 'int',
  'agencyid': 'double',
  'agencyname': 'string',
  'structype': 'string',
  'assetgroup': 'string',
  'assetcode': 'string',
  'assetdesc': 'string',
  'lga': 'string',
  'portauth': 'string',
  'sourcedate': 'dateTime',
  'source': 'string',
  'conmat': 'string',
  'constrdate': 'dateTime',
  'status': 'string',
  'editdate': 'dateTime',
  'techdraw': 'string',
  'refcomm': 'string',
  'reflink': 'string',
  'ci_id': 'string',
  'SourceDate_txt': 'string',
  'EditDate_txt': 'string',
  'st_perimeter_shape_': 'double'},
 'required': ['objectid', 'st_perimeter_shape_'],
 'geometry': '3D MultiPolygon',
 'geometry_column': 'shape'}

Next, as we have completed exploring the metadata related information of each dataset, we can now save them into geojson files.

In [71]:
# function to fetch + save + load
def fetch_and_save(layer_name):
    print(f"\nFetching {layer_name}...")

    response = wfs.getfeature(
        typename=layer_name,
        outputFormat='GEOJSON' 
    )

    file_path = f"../data/raw/{layer_name.replace(':', '_')}.geojson"

    # save raw file
    with open(file_path, 'wb') as f:
        f.write(response.read())

    response.close()

    print(f"Saved → {file_path}")

    # load into GeoDataFrame
    gdf = gpd.read_file(file_path)

    print(f"Loaded {len(gdf)} features")
    print(f"CRS: {gdf.crs}")
    print(f"Geometry types: {gdf.geometry.geom_type.unique()}")

    return gdf


# fetch all datasets
datasets = {}

for layer in layer_names:
    datasets[layer] = fetch_and_save(layer)


# assign for convenience
dot18 = datasets['esri:Coastal_Infrastructure_Point__DOT-018_']
dot19 = datasets['esri:Coastal_Infrastructure_Line__DOT-019_']
dot20 = datasets['esri:Coastal_Infrastructure_DOT__DOT-020_']


Fetching esri:Coastal_Infrastructure_Point__DOT-018_...
Saved → ../data/raw/esri_Coastal_Infrastructure_Point__DOT-018_.geojson
Loaded 5981 features
CRS: EPSG:4326
Geometry types: <StringArray>
['Point']
Length: 1, dtype: str

Fetching esri:Coastal_Infrastructure_Line__DOT-019_...
Saved → ../data/raw/esri_Coastal_Infrastructure_Line__DOT-019_.geojson
Loaded 50 features
CRS: EPSG:4326
Geometry types: [None]

Fetching esri:Coastal_Infrastructure_DOT__DOT-020_...
Saved → ../data/raw/esri_Coastal_Infrastructure_DOT__DOT-020_.geojson
Loaded 8535 features
CRS: EPSG:4326
Geometry types: <StringArray>
['MultiPolygon']
Length: 1, dtype: str


In [84]:
# initial data exploration
print(dot18.head)
print(dot18.columns)
print(dot18.dtypes)


#checking validity of datasets and if all values are correct/are present with no errors.
print(dot18.geometry.geom_type.value_counts())
print(dot18.geometry.is_valid.value_counts())
print(dot18.geometry.is_empty.sum())

<bound method NDFrame.head of                                             GmlID  objectid  agencyid  \
0        Coastal_Infrastructure_Point__DOT-018_.1         1       NaN   
1        Coastal_Infrastructure_Point__DOT-018_.2         2       NaN   
2        Coastal_Infrastructure_Point__DOT-018_.3         3       NaN   
3        Coastal_Infrastructure_Point__DOT-018_.4         4       NaN   
4        Coastal_Infrastructure_Point__DOT-018_.5         5       NaN   
...                                           ...       ...       ...   
5976  Coastal_Infrastructure_Point__DOT-018_.5977      5977       NaN   
5977  Coastal_Infrastructure_Point__DOT-018_.5978      5978       NaN   
5978  Coastal_Infrastructure_Point__DOT-018_.5979      5979       NaN   
5979  Coastal_Infrastructure_Point__DOT-018_.5980      5980       NaN   
5980  Coastal_Infrastructure_Point__DOT-018_.5981      5981       NaN   

     agencyname structype assetgroup assetcode  \
0          null       MRP         MR       

In [88]:
print(dot18.crs)
print(dot19.crs)
print(dot20.crs)
# all datasets are in same coordinate system. As CRS is in EPSG:4326, we need to reproject to another coordinate system.

EPSG:4326
EPSG:4326
EPSG:4326


In [93]:
print(dot18.describe(include='all'))
print(dot18['structype'].value_counts()) # what do these different structures mean?

                                           GmlID     objectid     agencyid  \
count                                       5981  5981.000000   286.000000   
unique                                      5981          NaN          NaN   
top     Coastal_Infrastructure_Point__DOT-018_.1          NaN          NaN   
freq                                           1          NaN          NaN   
mean                                         NaN  2991.000000  2845.573427   
std                                          NaN  1726.710312   314.135997   
min                                          NaN     1.000000  2342.000000   
25%                                          NaN  1496.000000  2590.250000   
50%                                          NaN  2991.000000  2795.000000   
75%                                          NaN  4486.000000  2971.250000   
max                                          NaN  5981.000000  3556.000000   

       agencyname structype assetgroup assetcode        assetde

In [100]:
# we need to sample plot test it

dot18.plot
base = dot18.plot(color='blue')
infra.plot(ax=base, color='red')



ImportError: The matplotlib package is required for plotting in geopandas. You can install it using 'conda install -c conda-forge matplotlib' or 'pip install matplotlib'.

In [103]:
# are we missing any data? No.

dot18.isnull().sum()

GmlID                0
objectid             0
agencyid          5695
agencyname           0
structype            0
assetgroup           0
assetcode            0
assetdesc            0
lga                  0
portauth             0
sourcedate           0
source               0
conmat               0
constrdate           0
status               0
editdate             0
techdraw             0
refcomm              0
reflink              0
ci_id                0
SourceDate_txt       0
EditDate_txt         0
geometry             0
dtype: int64

In [104]:
len(dot18)
dot18.memory_usage(deep=True)

Index                132
GmlID             549145
objectid           23924
agencyid           47848
agencyname        322484
structype         311120
assetgroup        305031
assetcode         305031
assetdesc         442007
lga               407863
portauth          321364
sourcedate        342608
source            661181
conmat            316993
constrdate        316993
status            340628
editdate          343571
techdraw          316993
refcomm           316993
reflink           316993
ci_id             334936
SourceDate_txt    346632
EditDate_txt      347476
geometry           47848
dtype: int64

we did some basic exploration especially of dot18 dataset. what we need to do next is the following.
1. make 1 notebook exploring each dataset (we also need to download the shapefile of the erosionhotspots and explore that).
2. in the epxloration stage do the basic exploration as we did above.
3. additional exploration you need to do is understand what the purpose of each dataset is and explore each variable
4. we need to convert CRS
5. sample plotting of dataset, try this out.
6. undeerstadn what each code name in the variable stands for.we need to find this out.